# CLIP + AST + fixed mapping — development 6000

Development-only decision-fusion baseline. AST event scores are mapped to scene
scores with the same fixed mapping used for the PANNs comparison, then combined
with CLIP over the original alpha sweep. The numerical procedure is unchanged.

## Setup

In [6]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
PROJECT_ROOT = Path('C:\\Users\\22313906t\\Desktop\\TAU_project')
CLIP_PATH = PROJECT_ROOT / 'results_CLIP' / 'tau_6000_clip_predictions.csv'
AST_PATH = PROJECT_ROOT / 'results_AST_6000' / 'tau_all_ast_events.csv'
FUSION_DIR = PROJECT_ROOT / 'results_FUSION_6000'
FUSION_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = FUSION_DIR / 'tau_6000_clip_ast_naive.csv'
SWEEP_PATH = FUSION_DIR / 'tau_6000_clip_ast_alpha_sweep.csv'
PER_CLASS_PATH = FUSION_DIR / 'tau_6000_clip_ast_per_class.csv'
TAU_CLASSES = [
    'airport',
    'bus',
    'metro',
    'metro station',
    'park',
    'public square',
    'shopping mall',
    'pedestrian street',
    'street traffic',
    'tram',
]
CLIP_SCORE_COLS = {scene: scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES}


def norm_event(x):
    x = str(x).lower()
    x = re.sub('[^a-z0-9]+', ' ', x)
    return re.sub('\\s+', ' ', x).strip()


## Fixed event-to-scene mapping

In [7]:
AUDIO_SCENE_MAP = {
    'bus': {'bus': 1.0},
    'air brake': {'bus': 0.9, 'street traffic': 0.4},
    'vehicle': {'bus': 0.55, 'metro': 0.4, 'metro station': 0.3, 'street traffic': 0.75, 'tram': 0.55},
    'motor vehicle road': {'bus': 0.25, 'street traffic': 1.0, 'tram': 0.15},
    'car': {'bus': 0.2, 'street traffic': 1.0, 'tram': 0.1},
    'engine': {'bus': 0.3, 'street traffic': 0.75, 'tram': 0.15},
    'vehicle horn car horn honking': {'bus': 0.2, 'street traffic': 0.9, 'tram': 0.15},
    'train': {'metro': 1.0, 'metro station': 0.9, 'tram': 0.7},
    'rail transport': {'metro': 1.0, 'metro station': 0.9, 'tram': 0.75},
    'railroad car train wagon': {'metro': 1.0, 'metro station': 0.85, 'tram': 0.7},
    'railroad car': {'metro': 1.0, 'metro station': 0.85, 'tram': 0.7},
    'train wheels squealing': {'metro': 0.85, 'metro station': 1.0, 'tram': 0.65},
    'subway metro underground': {'metro': 1.0, 'metro station': 1.0, 'tram': 0.15},
    'sliding door': {'bus': 0.3, 'metro': 0.45, 'metro station': 0.65, 'tram': 0.45},
    'door': {'bus': 0.15, 'metro': 0.2, 'metro station': 0.35, 'tram': 0.2},
    'bird': {'park': 1.0, 'public square': 0.1, 'pedestrian street': 0.1},
    'bird vocalization bird call bird song': {
        'park': 1.0,
        'public square': 0.1,
        'pedestrian street': 0.1,
    },
    'outside urban or manmade': {
        'public square': 0.8,
        'pedestrian street': 0.8,
        'street traffic': 0.55,
        'park': 0.2,
    },
    'aircraft': {'airport': 1.0},
    'aircraft engine': {'airport': 0.9},
    'jet engine': {'airport': 0.9},
    'fixed wing aircraft airplane': {'airport': 1.0},
}


## Load and prepare predictions

In [8]:
clip_df = pd.read_csv(CLIP_PATH)
clip_df['clip_id'] = clip_df['clip_id'].astype(str)
print('=' * 70)
print('CLIP + AST')
print('=' * 70)
print('CLIP clips:', clip_df['clip_id'].nunique())
for scene, col in CLIP_SCORE_COLS.items():
    if col not in clip_df.columns:
        raise ValueError(f'Missing CLIP score column: {col}')
experiment_ids = set(clip_df['clip_id'])
ast_df = pd.read_csv(AST_PATH)
ast_df['clip_id'] = ast_df['clip_id'].astype(str)
possible_label_cols = ['label', 'event', 'event_label', 'audio_event', 'class_name']
EVENT_COL = None
for col in possible_label_cols:
    if col in ast_df.columns:
        EVENT_COL = col
        break
if EVENT_COL is None:
    raise ValueError(f'Could not find AST event-label column.\nAvailable columns:\n{list(ast_df.columns)}')
possible_score_cols = ['max_score', 'score', 'mean_score']
SCORE_COL = None
for col in possible_score_cols:
    if col in ast_df.columns:
        SCORE_COL = col
        break
if SCORE_COL is None:
    raise ValueError(f'Could not find AST score column.\nAvailable columns:\n{list(ast_df.columns)}')
print('AST event column:', EVENT_COL)
print('AST score column:', SCORE_COL)
ast_df = ast_df[ast_df['clip_id'].isin(experiment_ids)].copy()
print('AST matched clips:', ast_df['clip_id'].nunique())
print('AST event rows:', len(ast_df))
ast_df['event_norm'] = ast_df[EVENT_COL].apply(norm_event)
audio_rows = []
for clip_id, group in ast_df.groupby('clip_id'):
    raw_scores = {scene: 0.0 for scene in TAU_CLASSES}
    for _, row in group.iterrows():
        event = row['event_norm']
        try:
            event_score = float(row[SCORE_COL])
        except:
            continue
        if not np.isfinite(event_score):
            continue
        if event not in AUDIO_SCENE_MAP:
            continue
        for scene, weight in AUDIO_SCENE_MAP[event].items():
            raw_scores[scene] += event_score * weight
    total = sum(raw_scores.values())
    if total > 0:
        scores = {scene: raw_scores[scene] / total for scene in TAU_CLASSES}
        has_audio = True
    else:
        scores = {scene: 0.0 for scene in TAU_CLASSES}
        has_audio = False
    out = {'clip_id': clip_id, 'has_audio_evidence': has_audio}
    for scene in TAU_CLASSES:
        out['audio_' + scene.replace(' ', '_') + '_score'] = scores[scene]
    audio_rows.append(out)
audio_df = pd.DataFrame(audio_rows)
fusion_df = clip_df.merge(audio_df, on='clip_id', how='left')
audio_score_cols = ['audio_' + scene.replace(' ', '_') + '_score' for scene in TAU_CLASSES]
fusion_df[audio_score_cols] = fusion_df[audio_score_cols].fillna(0.0)
fusion_df['has_audio_evidence'] = fusion_df['has_audio_evidence'].fillna(False)
print(
    '\nClips with mapped AST evidence:',
    int(fusion_df['has_audio_evidence'].sum()),
    '/',
    len(fusion_df),
)
clip_accuracy = accuracy_score(fusion_df['ground_truth'], fusion_df['clip_top1'])
clip_macro_f1 = f1_score(
    fusion_df['ground_truth'],
    fusion_df['clip_top1'],
    labels=TAU_CLASSES,
    average='macro',
    zero_division=0,
)
print('\nOriginal CLIP accuracy:', f'{clip_accuracy * 100:.2f}%')
print('Original CLIP Macro-F1:', f'{clip_macro_f1:.4f}')


CLIP + AST
CLIP clips: 6000
AST event column: label
AST score column: max_score
AST matched clips: 6000
AST event rows: 97116

Clips with mapped AST evidence: 5956 / 6000

Original CLIP accuracy: 78.93%
Original CLIP Macro-F1: 0.7877


## Fusion-weight sweep

In [9]:
def evaluate_alpha(df, alpha):
    predictions = []
    top_scores = []
    for _, row in df.iterrows():
        clip_scores = np.array([float(row[CLIP_SCORE_COLS[scene]]) for scene in TAU_CLASSES])
        audio_scores = np.array([float(row['audio_' + scene.replace(' ', '_') + '_score']) for scene in TAU_CLASSES])
        if audio_scores.sum() <= 0:
            fused = clip_scores
        else:
            fused = alpha * clip_scores + (1.0 - alpha) * audio_scores
        best = int(np.argmax(fused))
        predictions.append(TAU_CLASSES[best])
        top_scores.append(float(fused[best]))
    acc = accuracy_score(df['ground_truth'], predictions)
    mf1 = f1_score(df['ground_truth'], predictions, labels=TAU_CLASSES, average='macro', zero_division=0)
    return (predictions, top_scores, acc, mf1)


ALPHAS = [0.95, 0.9, 0.85, 0.8, 0.75, 0.7, 0.65, 0.6]
rows = []
for alpha in ALPHAS:
    predictions, scores, accuracy, macro_f1 = evaluate_alpha(fusion_df, alpha)
    rows.append({'clip_weight_alpha': alpha, 'audio_weight': 1.0 - alpha, 'accuracy': accuracy, 'accuracy_percent': accuracy * 100, 'macro_f1': macro_f1, 'improvement_vs_clip_points': (accuracy - clip_accuracy) * 100})
sweep_df = pd.DataFrame(rows)
sweep_df.to_csv(SWEEP_PATH, index=False)
print('\n' + '=' * 70)
print('CLIP + AST ALPHA SWEEP')
print('=' * 70)
print(sweep_df.round(4).to_string(index=False))
best_row = sweep_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
BEST_ALPHA = float(best_row['clip_weight_alpha'])



CLIP + AST ALPHA SWEEP
 clip_weight_alpha  audio_weight  accuracy  accuracy_percent  macro_f1  improvement_vs_clip_points
              0.95          0.05    0.7895           78.9500    0.7879                      0.0167
              0.90          0.10    0.7902           79.0167    0.7886                      0.0833
              0.85          0.15    0.7915           79.1500    0.7898                      0.2167
              0.80          0.20    0.7925           79.2500    0.7909                      0.3167
              0.75          0.25    0.7930           79.3000    0.7915                      0.3667
              0.70          0.30    0.7940           79.4000    0.7925                      0.4667
              0.65          0.35    0.7942           79.4167    0.7925                      0.4833
              0.60          0.40    0.7932           79.3167    0.7913                      0.3833


## Final evaluation and outputs

In [10]:
predictions, scores, best_accuracy, best_macro_f1 = evaluate_alpha(fusion_df, BEST_ALPHA)
fusion_df['ast_fused_top1'] = predictions
fusion_df['ast_fused_top1_score'] = scores
fusion_df['clip_correct'] = fusion_df['clip_top1'] == fusion_df['ground_truth']
fusion_df['fusion_correct'] = fusion_df['ast_fused_top1'] == fusion_df['ground_truth']
fusion_df[
    'change_type'] = np.select([~fusion_df['clip_correct'] & fusion_df['fusion_correct'], fusion_df['clip_correct'] & ~fusion_df['fusion_correct'], fusion_df['clip_correct'] & fusion_df['fusion_correct']],
    ['Wrong->Correct', 'Correct->Wrong', 'Correct->Correct'],
    default='Wrong->Wrong',
)
change_counts = fusion_df['change_type'].value_counts()
per_class_rows = []
for scene in TAU_CLASSES:
    temp = fusion_df[fusion_df['ground_truth'] == scene]
    clip_acc = (temp['clip_top1'] == scene).mean()
    fused_acc = (temp['ast_fused_top1'] == scene).mean()
    per_class_rows.append({'class': scene, 'number_of_clips': len(temp), 'clip_accuracy_percent': clip_acc * 100, 'clip_ast_accuracy_percent': fused_acc * 100, 'change_points': (fused_acc - clip_acc) * 100})
per_class_df = pd.DataFrame(per_class_rows)
per_class_df.to_csv(PER_CLASS_PATH, index=False)
fusion_df.to_csv(OUTPUT_PATH, index=False)
print('\n' + '=' * 70)
print('BEST EXPLORATORY CLIP + AST RESULT')
print('=' * 70)
print('Best CLIP weight:', BEST_ALPHA)
print('AST weight:', round(1.0 - BEST_ALPHA, 2))
print('\nCLIP:', f'{clip_accuracy * 100:.2f}%')
print('CLIP + AST:', f'{best_accuracy * 100:.2f}%')
print('Improvement:', f'{(best_accuracy - clip_accuracy) * 100:+.2f}', 'percentage points')
print('\nCLIP Macro-F1:', f'{clip_macro_f1:.4f}')
print('Fusion Macro-F1:', f'{best_macro_f1:.4f}')
print('\nCorrection analysis:')
for label in ['Wrong->Correct', 'Correct->Wrong', 'Correct->Correct', 'Wrong->Wrong']:
    print(f'{label:18s}:', int(change_counts.get(label, 0)))
print('\nPer-class results:')
print(per_class_df.round(2).to_string(index=False))
print('\nSaved:')
print(OUTPUT_PATH)
print(SWEEP_PATH)
print(PER_CLASS_PATH)
print('\nIMPORTANT:')
print('This alpha sweep is exploratory only.')
print('For the paper, select alpha on VALIDATION and evaluate once on TEST.')



BEST EXPLORATORY CLIP + AST RESULT
Best CLIP weight: 0.65
AST weight: 0.35

CLIP: 78.93%
CLIP + AST: 79.42%
Improvement: +0.48 percentage points

CLIP Macro-F1: 0.7877
Fusion Macro-F1: 0.7925

Correction analysis:
Wrong->Correct    : 63
Correct->Wrong    : 34
Correct->Correct  : 4702
Wrong->Wrong      : 1201

Per-class results:
            class  number_of_clips  clip_accuracy_percent  clip_ast_accuracy_percent  change_points
          airport              600                  80.33                      79.33          -1.00
              bus              600                  54.67                      58.50           3.83
            metro              600                  96.33                      96.50           0.17
    metro station              600                  74.17                      74.00          -0.17
             park              600                  89.83                      90.67           0.83
    public square              600                  67.00            